# 02 — Logs: collection & visualization

Query Loki from the notebook, parse the response into pandas, and visualize log volume + a rough error rate.

What you'll do:

1. Hit the Loki HTTP API directly (`/loki/api/v1/query_range`) for a LogQL window.
2. Use the bundled `logcli` shell command for ad-hoc tailing and one-off queries.
3. Parse JSON-formatted log lines with `pandas.json_normalize` and chart per-namespace volume.
4. List Loki ruler groups with `lokitool` to see what alerts are deployed.

In [ ]:
import os

LOKI_URL = os.environ.get('LOKI_URL', os.environ.get('LOKI_ADDRESS', 'http://loki:3100'))
LOKI_TENANT = os.environ.get('LOKI_TENANT_ID', '')
LOGQL = '{job=~".+"}'   # narrow to a real selector before running
RANGE_HOURS = 1
LIMIT = 1000

In [ ]:
import time, json, requests, pandas as pd

def _headers():
    return {'X-Scope-OrgID': LOKI_TENANT} if LOKI_TENANT else {}

def loki_query_range(logql, hours=RANGE_HOURS, limit=LIMIT):
    end_ns = int(time.time() * 1e9)
    start_ns = end_ns - hours * 3600 * 10**9
    r = requests.get(f'{LOKI_URL}/loki/api/v1/query_range',
                     params={'query': logql, 'start': start_ns, 'end': end_ns,
                             'limit': limit, 'direction': 'BACKWARD'},
                     headers=_headers(), timeout=60)
    r.raise_for_status()
    rows = []
    for stream in r.json()['data']['result']:
        labels = stream['stream']
        for ts, line in stream['values']:
            rows.append({**labels, 't': pd.to_datetime(int(ts), unit='ns'), 'line': line})
    return pd.DataFrame(rows)

df = loki_query_range(LOGQL)
print(f'rows: {len(df)}')
df.head(3)

## Volume per minute, by job

In [ ]:
import matplotlib.pyplot as plt

if not df.empty:
    grp = df.set_index('t').groupby([pd.Grouper(freq='1min'), 'job']).size().unstack(fill_value=0)
    grp.plot(figsize=(11, 4), title='Log lines / minute by job', legend=True)
    plt.tight_layout()

## Parse JSON log lines

If your apps log structured JSON, `pandas.json_normalize` flattens it into columns you can group/aggregate.

In [ ]:
def safe_json(s):
    try:
        return json.loads(s)
    except Exception:
        return {}

if not df.empty:
    parsed = pd.json_normalize(df['line'].map(safe_json))
    parsed.head(5)

## Error rate (LogQL metric query)

LogQL can return numeric series — same shape as Prometheus — by wrapping a stream selector in `rate()` / `count_over_time()`. Use this for graphs, not row pulls.

In [ ]:
def loki_metric_query_range(metric_logql, hours=RANGE_HOURS, step='30s'):
    end = int(time.time())
    start = end - hours * 3600
    r = requests.get(f'{LOKI_URL}/loki/api/v1/query_range',
                     params={'query': metric_logql, 'start': start, 'end': end, 'step': step},
                     headers=_headers(), timeout=60)
    r.raise_for_status()
    rows = []
    for s in r.json()['data']['result']:
        for ts, val in s['values']:
            rows.append({**s['metric'], 't': pd.to_datetime(ts, unit='s'), 'v': float(val)})
    return pd.DataFrame(rows)

err = loki_metric_query_range('sum by (job) (rate({job=~".+"} |= "error" [1m]))')
if not err.empty:
    err.pivot_table(index='t', columns='job', values='v').plot(figsize=(11, 4), title='Error log rate by job')
    plt.tight_layout()

## CLI: `logcli` for ad-hoc queries

`logcli` is on `$PATH` in the image — great for quick tailing or piped output.

In [ ]:
!LOKI_ADDR=$LOKI_URL LOKI_ORG_ID=$LOKI_TENANT logcli query --limit 5 '$LOGQL' 2>&1 | head -20

## Inspect Loki ruler with `lokitool`

In [ ]:
!LOKI_ADDRESS=$LOKI_URL LOKI_TENANT_ID=$LOKI_TENANT lokitool rules list 2>&1 | head -40

### Where to go next

- Promote useful selectors to Loki ruler groups under a mixin's `lokiRules`/`lokiAlerts` block.
- Cross-reference logs with metrics by aligning on `pod`/`namespace` labels in your Loki and Prometheus selectors.